In [ ]:
import json
import os
import random

import numpy as np
import torch
import torchvision.transforms.v2 as tfs

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

In [ ]:
base_dir = os.path.join(os.getcwd(), 'asl_alphabet_train')
base_dir

In [ ]:
classes = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
print(classes)

['A', 'B', 'C', 'D', 'del', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'nothing', 'O', 'P', 'Q', 'R', 'S', 'space', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [ ]:
classes = classes[:3]
classes

['A', 'B', 'C']

In [ ]:
split_ratio = 0.2
file_format = 'format.json'
# to_tensor = tfs.ToTensor()
to_tensor = tfs.Compose([tfs.ToImage(), tfs.ToDtype(torch.float32, scale=True)])
transform = tfs.ToPILImage()

for cls in classes:
        cls_source_path = os.path.join(base_dir, cls)
        cls_train_path = os.path.join('dataset', 'train', f'class_{cls}')
        cls_test_path = os.path.join('dataset', 'test', f'class_{cls}')

        # Создаем папки для классов в train и test
        os.makedirs(cls_train_path, exist_ok=True)
        os.makedirs(cls_test_path, exist_ok=True)

        # Получаем список файлов в классе
        files = [f for f in os.listdir(cls_source_path) if os.path.isfile(os.path.join(cls_source_path, f))]

        # Перемешиваем файлы
        random.shuffle(files)
        
        # Определяем количество файлов, которые будут зайдействованы
        per_class = len(files) // 6 # 500

        files = files[:per_class]

        # Определяем количество файлов для train и test
        split_index = int(len(files) * split_ratio)

        train_files = files[:per_class][split_index:]
        test_files = files[:per_class][:split_index]

        def process_and_save(src_path, dst_path):
          img = Image.open(src_path).convert("RGB")
          img_tensor = to_tensor(img)
          img_transformed = transform(img_tensor)
          img_transformed.save(dst_path, "PNG")

        for f in train_files:
            src_file = os.path.join(cls_source_path, f)
            dst_file = os.path.join(cls_train_path, os.path.splitext(f)[0] + ".png")
            process_and_save(src_file, dst_file)

        for f in test_files:
            src_file = os.path.join(cls_source_path, f)
            dst_file = os.path.join(cls_test_path, os.path.splitext(f)[0] + ".png")
            process_and_save(src_file, dst_file)

        print(f"Класс {cls}: всего {len(files)} файлов, train={len(train_files)}, test={len(test_files)}")

Класс A: всего 500 файлов, train=400, test=100
Класс B: всего 500 файлов, train=400, test=100
Класс C: всего 500 файлов, train=400, test=100


In [ ]:
targets = dict()
for i, cls in enumerate(classes):
    targets[f'class_{cls}'] = cls

fp = open(os.path.join(os.getcwd(), 'dataset', file_format), "w")
json.dump(targets, fp)
fp.close()